# 4.2: Skip-gram with negative sampling in PyTorch

Putting the pieces together, the whole model is two embedding tables and the loss above. `nn.Embedding` is PyTorch's lookup table from Section 4.1: `emb(ids)` fetches rows `ids` of the table, and only those rows receive gradients.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.

This snippet uses `noise_dist`, `num_steps`, `pairs` from the surrounding training code. Those names are not defined in an earlier listing, so the cell shows the pattern as it appears in the chapter.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From One-hot vectors
import numpy as np

vocab = ["cat", "dog", "car", "banana"]

one_hot = np.eye(len(vocab))

def cos(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

print("cos(cat, dog)    =", cos(one_hot[0], one_hot[1]))

print("cos(cat, banana) =", cos(one_hot[0], one_hot[3]))

print("distance(cat, dog)    =", np.linalg.norm(one_hot[0] - one_hot[1]).round(4))

print("distance(cat, banana) =", np.linalg.norm(one_hot[0] - one_hot[3]).round(4))

# From Gradients
import torch

import torch.nn.functional as F

d, k = 8, 5

v_c = torch.randn(d, requires_grad=True)        # center word, input vector

u_o = torch.randn(d, requires_grad=True)        # true context word, output vector

u_neg = torch.randn(k, d, requires_grad=True)   # k negative samples, output vectors

loss = -F.logsigmoid(u_o @ v_c) - F.logsigmoid(-(u_neg @ v_c)).sum()

loss.backward()

with torch.no_grad():                           # the formulas derived above
    s_pos = torch.sigmoid(u_o @ v_c)
    s_neg = torch.sigmoid(u_neg @ v_c)
    g_vc = (s_pos - 1) * u_o + (s_neg[:, None] * u_neg).sum(0)
    g_uo = (s_pos - 1) * v_c
    g_neg = s_neg[:, None] * v_c

print(f"loss = {loss.item():.4f}")

for name, ours, auto in [("v_c", g_vc, v_c.grad), ("u_o", g_uo, u_o.grad), ("u_neg", g_neg, u_neg.grad)]:
    print(f"{name:5s}: max |difference| = {(ours - auto).abs().max().item():.1e}")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SkipGramNS(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.inp = nn.Embedding(vocab_size, dim)    # v_w: center-word vectors
        self.out = nn.Embedding(vocab_size, dim)    # u_w: context-word vectors
        nn.init.uniform_(self.inp.weight, -0.5 / dim, 0.5 / dim)
        nn.init.zeros_(self.out.weight)

    def forward(self, center, context, negatives):
        # center: (B,), context: (B,), negatives: (B, k), all integer word IDs
        v = self.inp(center)                                  # (B, d)
        u_pos = self.out(context)                             # (B, d)
        u_neg = self.out(negatives)                           # (B, k, d)
        pos = F.logsigmoid((u_pos * v).sum(-1))               # (B,)
        neg = F.logsigmoid(-(u_neg @ v.unsqueeze(-1)).squeeze(-1)).sum(-1)  # (B,)
        return -(pos + neg).mean()


A training step draws a minibatch of (center, context) pairs, samples $`k`$ negatives per pair from the noise distribution with `torch.multinomial`, and runs the usual loop of Section 2.9:


In [ ]:
# Assumes vocab, pairs (a LongTensor of (center, context) IDs), noise_dist, k, and num_steps
model = SkipGramNS(len(vocab), dim=100)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
for step in range(num_steps):
    batch = pairs[torch.randint(len(pairs), (256,))]                  # (256, 2)
    negatives = torch.multinomial(noise_dist, 256 * k, replacement=True).view(256, k)
    loss = model(batch[:, 0], batch[:, 1], negatives)
    opt.zero_grad()
    loss.backward()
    opt.step()
